## Google Pay Expense Sharing

##### This is a Python program for splitting shared expenses in a group. It tracks who paid, calculates who owes money, and suggests who should pay whom to settle everything.


## IMPORTS

In [7]:
import numpy as np
from prettytable import PrettyTable

## COLLEAGUE LIST

In [8]:
colleague=["Marley","Chaplin","Joseph","Peter"]
expense_matrix=np.zeros((len(colleague),len(colleague)))
transaction_history = []

## EQUAL SPLIT

In [9]:
def add_expense(description,payer,beneficiaries,amount):
    payer_index = colleague.index(payer)
    share_per_person =  amount / len(beneficiaries)
    
    for beneficiary in beneficiaries:
        beneficiary_index = colleague.index(beneficiary)
        expense_matrix[payer_index][beneficiary_index]+=share_per_person

    transaction_history.append({
        
        "description": description,
        "payer": payer,
        "amount": amount,
        "type": "Equal Split"
    })

## CUSTOM SPLIT

In [10]:
def add_custom_expense(description, payer, splits):
   
    payer_index = colleague.index(payer)
    total_amount = sum(splits.values())

    for person, amount in splits.items():
        person_index = colleague.index(person)
        expense_matrix[payer_index][person_index] += amount

    transaction_history.append({
        "description": description,
        "payer": payer,
        "amount": total_amount,
        "type": "Custom Split"
    })

In [29]:
add_expense(
    "Lunch",
    "Marley",
    ["Marley", "Chaplin", "Joseph"],
    1200)

add_expense(
    "Snacks",
    "Chaplin",
    ["Chaplin", "Joseph"],
    845)

add_expense(
    "Dinner",
    "Joseph",
    ["Marley", "Chaplin", "Joseph", "Peter"],
    1700)

add_custom_expense("Tea","Peter",{"Marley":30,"Chaplin":150,"Joseph":200})
add_custom_expense("Movie", "Marley", {"Chaplin": 150, "Joseph": 100, "Peter": 200})

## SETTLEMENT CALCULATION

In [12]:
def calculate_settlement():
    total_paid = np.sum(expense_matrix,axis=1)
    total_owed=np.sum(expense_matrix,axis=0)
    net_balance=total_paid-total_owed
    return net_balance

## DISPLAY SETTLEMENT

In [13]:
def display_settlement():
    settlements=calculate_settlement()

    table=PrettyTable()
    table.field_names=["Colleague","Settlement"]

    for i,person in enumerate(colleague):
        if settlements[i] > 0:
            table.add_row([
                person,
                f"Should receive ₹{settlements[i]:.2f}"])

        elif settlements[i] < 0:
            table.add_row([
                person, 
                f"Owes ₹{-settlements[i]:.2f}"])

        else:
            table.add_row([person, "Is settled"])

    print("\nFinal Settlement:")
    print(table)

## SUGGESTED PAYMENTS

In [14]:
def suggest_payments():
    settlements = calculate_settlement()

    creditors = [(colleague[i], amt)
                 for i, amt in enumerate(settlements)
                 if amt > 0]

    debtors = [(colleague[i], -amt)
               for i, amt in enumerate(settlements)
               if amt < 0]

    transactions = []
    

    # Match debtors with creditors
    while debtors and creditors:

        debtor, debt_amount = debtors.pop(0)
        creditor, credit_amount = creditors.pop(0)

        payment = min(debt_amount, credit_amount)

        transactions.append((debtor, creditor, payment))

        # Update balances
        debt_amount -= payment
        credit_amount -= payment

        # Put back if money remains
        if debt_amount > 0:
            debtors.insert(0, (debtor, debt_amount))

        if credit_amount > 0:
            creditors.insert(0, (creditor, credit_amount))

    print("\nSuggested Transactions:")

    if transactions:
        for debtor, creditor, amount in transactions:
            print(f"{debtor} should pay ₹{amount:.2f} to {creditor}")
    else:
        print("No transactions needed. Everyone is settled.")

## TRANSACTION HISTORY

In [15]:
def show_history():
    print("\nTransaction History (All)")
    for transaction in transaction_history:
        print(f"Description : {transaction['description']}")
        print(f"Payer       : {transaction['payer']}")
        print(f"Amount      : ₹{transaction['amount']}")
        print(f"Type        : {transaction['type']}")
        print("-" * 30)

In [22]:
def show_user_history(person):
    print(f"\nTransaction History for {person}")
    print("-" * 30)

    found = False
    for transaction in transaction_history:
        if transaction["payer"] == person or person in transaction.get("people", []):
            print(f"Description : {transaction['description']}")
            print(f"Payer       : {transaction['payer']}")
            print(f"Amount      : ₹{transaction['amount']}")
            print(f"Type        : {transaction['type']}")
            print("-" * 30)
            found = True

    if not found:
        print("No transactions found.")

In [30]:
#display final settlement
display_settlement()
# Who pays whom
suggest_payments()


Final Settlement:
+-----------+-------------------------+
| Colleague |        Settlement       |
+-----------+-------------------------+
|   Marley  | Should receive ₹2265.00 |
|  Chaplin  |      Owes ₹2462.50      |
|   Joseph  | Should receive ₹1762.50 |
|   Peter   |      Owes ₹1565.00      |
+-----------+-------------------------+

Suggested Transactions:
Chaplin should pay ₹2265.00 to Marley
Chaplin should pay ₹197.50 to Joseph
Peter should pay ₹1565.00 to Joseph


In [31]:
# All transactions together
show_history()


Transaction History (All)
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Snacks
Payer       : Chaplin
Amount      : ₹845
Type        : Equal Split
------------------------------
Description : Dinner
Payer       : Joseph
Amount      : ₹1700
Type        : Equal Split
------------------------------
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Snacks
Payer       : Chaplin
Amount      : ₹845
Type        : Equal Split
------------------------------
Description : Dinner
Payer       : Joseph
Amount      : ₹1700
Type        : Equal Split
------------------------------
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Snacks
Payer       : Chaplin
Amount      : ₹845
Type        : Equal Split
------------------------------
Description : Dinner
Pay

In [32]:
# Each person's individual history
for person in colleague:
    show_user_history(person)


Transaction History for Marley
------------------------------
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Lunch
Payer       : Marley
Amount      : ₹1200
Type        : Equal Split
------------------------------
Description : Movie
Payer       : Marley
Amount      : ₹450
Type        : Custom Split
------------------------------

Transaction History for Chaplin
------------------------------
Description : Snacks
Payer       : Chaplin
Amount      : ₹845
Type        : Equal Split
------------------------------
Description : Snacks
Payer       : Chaplin
Amo